***

# Trabajo Biofísica: Clasificación y Correlación de las estadísticas en Pokémon

*** 

# Velasco García, Lucía
# Villaverde Turnes, David

***
## Introducción

Pokémon es una saga de videojuegos creada y desarrollada por *Game Freak* en 1996, que se popularizó rápidamente debido a su estilo sencillo de juego y progresión. El objetivo de estos juegos radica en convertirse en entrenador de unas criaturas llamadas pokémon. Para poder mejorar y continuar en la historia principal deberás usar tus pokémon para combatir contra otros entrenadores y así convertirte en el mejor.

Estas criaturas están agrupadas en algo que llamamos tipos, que representan el elemento que los caracteriza. Estos tipos van desde afinidades más habituales como fuego, agua o electricidad, hasta otras más hetéreas, como fantasma, dragón, etc. Cada pokémon puede tener uno o dos tipos.

A mayores de esto, cada pokémon se caracteriza por tener unas determinadas estadísticas, que representan cuán fuertes son. Estas estadísticas son: puntos de salud, ataque, defensa, ataque, defensa especiales y velocidad. 

Las estadísticas, en principio, no tienen por qué estar relacionadas con los tipos. Sin embargo, se cree que hay pares de tipos con estadísticas contrarias entre sí. Esto lo podemos observar, por ejemplo, en los tipos psíquico y lucha: los primeros se caracterizan por tener alto ataque y defensa (golpes 'de contacto') pero bajo ataque y defensa especial (ataques 'a distancia'), mientras que en los segundos ocurre justo lo contrario.

Nuestro objetivo en este trabajo será, en primer lugar, analizar la relación que puede haber entre las estadísticas de ataque-ataque especial y defensa-defensa especial de todos los pokémon. Después, escogeremos solo los que tengan tipo lucha o tipo psíquico y entrenaremos un modelo que sea capaz de predecir el tipo del pokémon a partir de sus estadísticas.

Como último ejercicio, vamos a utilizar otro tipo de clasificación entre todos los pokémon: si son o no legendarios. En los juegos, los pokémon legendarios se caracterizan por tener una fuerza y unas características extraordinarias. Vamos a comprobar si esto es verídico o no entrenando un modelo de red neuronal que analice todas las estadísticas de cada pokémon y que sea capaz de predecir si este es o no legendario.

***

Base de datos: https://www.kaggle.com/datasets/abcsds/pokemon?resource=download

***

***

## Pasos previos

Vamos a, en primer lugar, cargar nuestra base de datos y echar un vistazo a qué hay: las distintas categorías que encontramos, si hay o no valores duplicados o perdidos, etc.

***

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar los datos
df = pd.read_csv('trabajobio/Pokemon.csv')

# Información general
print('\n','=' * 50)
print(f"Total de Pokémon: {df.shape[0]}")
print(f"Total de variables: {df.shape[1]}")
print('\n','=' * 50)

print('\n','=' * 50)
print("\nColumnas del dataset:")
print(df.columns.tolist())
print('\n','=' * 50)

print('\n','=' * 50)
print("\nNúmero de Pokémon de cada tipo:")
print(df[['Type 1','Type 2']].stack().value_counts())
print('\n','=' * 50)

print('\n','=' * 50)
print(f'\n Número de Pokémon legendarios: {df["Legendary"].sum()}')
print(f'\n Número de Pokémon no legendarios: {df.shape[0] - df["Legendary"].sum()}')
print('\n','=' * 50)

display(df.head())



***

## Preparación y limpieza de los datos

Antes de entrenar la red neuronal, es necesario preparar el conjunto de datos para garantizar que la información utilizada sea adecuada.

En esta etapa se comprobarán las dimensiones del dataset, los tipos de variables, la existencia de duplicados y los posibles valores ausentes.

***

In [ ]:
# Dimensiones iniciales
print('\n','=' * 50)
print("Dimensiones iniciales:", df.shape)
print('\n','=' * 50)

# Primeras filas
print('\n','=' * 50)
display(df.head())
print('\n','=' * 50)

# Información de las variables
print('\n','=' * 50)
df.info()
print('\n','=' * 50)

# Eliminación de duplicados
df = df.drop_duplicates()

print('\n','=' * 50)
print("\nDimensiones tras eliminar duplicados:", df.shape)
print('\n','=' * 50)

# Eliminación de valores perdidos
df = df.dropna()

print('\n','=' * 50)
print("\nDimensiones tras eliminar valores perdidos:", df.shape)
print('\n','=' * 50)


***

Vemos que se pierden algunas filas, es decir, algunos pokémon. Aproximadamente la mitad. 


Como comentamos en la introducción, los pokémon pueden tener hasta dos tipos. Comprobaremos que lo que está ocurriendo es que el programa está eliminando los que solo tienen un tipo.

***


In [ ]:
#Volvemos a cargar el archivo de origen
df = pd.read_csv('trabajobio/Pokemon.csv')

print('\n' + '=' * 50)
print(df['Type 2'].unique())
print('\n' + '=' * 50)

print('\n' + '=' * 50)

restantes = df.shape[0] - df['Type 2'].isna().sum()
print(f"Número de Pokémon sin tipo secundario: {restantes}")

print('\n' + '=' * 50)

***

Como vemos, el número de pokémon que nos quedan son justamente los que quedaban después de eliminar los nans. 


Para poder trabajar más cómodamente con los datos, vamos a sustituir los 'nan' por 'None'.

***

In [ ]:
df['Type 2'] = df['Type 2'].fillna('None')
print('\n' + '=' * 50)
print(df['Type 2'].unique())
print('\n' + '=' * 50)

***

# Parte 1: SVR para evaluar las parejas de ataque y defensa físicos y especiales

*** 

Implementaremos un modelo SVR para detectar si hay cierta correlación entre las parejas de ataque-ataque especial y defensa-defensa especial. Escogeremos como variable X el ataque y evaluaremos la dependencia de Y, el ataque especial. Después, haremos lo mismo con las defensas.

***

Vamos a basarnos en el archivo de SVR que podemos encontrar en el aula virtual.

***

In [ ]:
#Preparamos en primer lugar los datos

X = df['Attack'].values.reshape(-1, 1)
Y = df['Sp. Atk'].values.reshape(-1, 1)

#Escalamos los datos para que estén entre 0 y 1, ya que el SVR es sensible a la escala de los datos

from sklearn.preprocessing import MinMaxScaler 
X_scaler = MinMaxScaler()
Y_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X) 
Y = Y_scaler.fit_transform(Y)

***

Continuaremos dividiendo los datos en conjuntos de entrenamiento, los cuales nos servirán para entrenar el modelo, y prueba, con los cuales se comprobará si este funciona correctamente.

***


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.33, random_state=100)

***
A continuación, tenemos que escoger los parámetros del modelo de regresión. Viendo los resultados que obtuvimos en clase y, como realmente no sabemos si hay una dependencia o no, vamos a escoger como kernel el 'rbf', que es el menos restrictivo y el que puede caracterizar más casos. Además, escogeremos $\gamma$=1, que es valor por defecto, para ser más permisivos con el modelo. Por último, igual que en clase, escogeremos C=100

***

In [ ]:
from sklearn.svm import SVR

svr_rbf = SVR(kernel='rbf', C=100, gamma=1, epsilon=.1) 

# Creando el modelo y entrenando
reg = svr_rbf.fit(X_train,Y_train.reshape(-1))

# Prediciendo valores de entrenamiento
Y_train_hat = svr_rbf.predict(X_train)

# Prediciendo valores de validación
Y_test_hat = svr_rbf.predict(X_test)



# Creamos un scatter plot con los datos de entrenamiento y validación
plt.scatter(X_train, Y_train)
plt.scatter(X_test, Y_test)

# En X_plot creamos un continuo de datos entre 0 y 1
X_plot = np.linspace(0,1).reshape(-1, 1)
Y_plot = svr_rbf.predict(X_plot)


# Graficamos el modelo
plt.plot(X_plot, Y_plot,"r--")
plt.xlabel('Ataque')
plt.ylabel('Ataque especial')
plt.legend(['Datos de entrenamiento', 'Datos de prueba', 'Modelo'])
plt.grid()
# Calculamos el error
from sklearn.metrics import r2_score
print("Entrenamiento", r2_score(Y_train, Y_train_hat))
print("Prueba", r2_score(Y_test, Y_test_hat))

***

Podemos observar como el modelo no es nada bueno. La correlación es muy baja. Además, probando con el modelo lineal da un poco peor.

Por lo tanto, concluímos que, en general, el ataque especial no depende del ataque. ¿Qué pasa con la defensa?

***


In [ ]:
#Hacemos todo exactamente igual pero con las variables de defensa y defensa especial

X = df['Defense'].values.reshape(-1, 1)
Y = df['Sp. Def'].values.reshape(-1, 1)


X_scaler = MinMaxScaler()
Y_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X) 
Y = Y_scaler.fit_transform(Y)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.33, random_state=100)



svr_rbf = SVR(kernel='rbf', C=100, gamma=1, epsilon=.1) 


reg = svr_rbf.fit(X_train,Y_train.reshape(-1))
Y_train_hat = svr_rbf.predict(X_train)
Y_test_hat = svr_rbf.predict(X_test)
plt.scatter(X_train, Y_train)
plt.scatter(X_test, Y_test)

X_plot = np.linspace(0,1).reshape(-1, 1)
Y_plot = svr_rbf.predict(X_plot)



plt.plot(X_plot, Y_plot,"r--")
plt.xlabel('Defense'); plt.ylabel('Sp. Def')
plt.legend(['Datos de entrenamiento', 'Datos de prueba', 'Modelo'])
plt.grid()

print("Entrenamiento", r2_score(Y_train, Y_train_hat))
print("Prueba", r2_score(Y_test, Y_test_hat))

***

Aunque con la defensa obtenemos un resultado un poco mejor, sigue sin ser gran cosa. Por tanto, podemos desmentir el hecho de que haya una relación entre estas estadísticas.

Ahora bien, ¿será entonces cierto que se distinguen el tipo lucha y psíquico?

***

***

# Parte 2: ¿Psíquico o lucha?

## Escogemos las variables que vamos a utilizar

Como comentamos anteriormente, vamos a intentar predecir el tipo de pokémon a partir de sus estadísiticas, restringiéndonos únicamente a los tipos psíquico y lucha. Para ello, nuestra variable X será todas las estadísticas del pokémon, mientras que la Y corresponderá a sus tipos (psíquico y lucha)

***

In [ ]:
# Variables que utilizaremos
estadísticas = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

# Seleccionamos únicamente Pokémon de tipo Psíquico y Lucha
df_filtrado = df[df['Type 1'].isin(['Psychic', 'Fighting']) | df['Type 2'].isin(['Psychic', 'Fighting'])].copy()


# Creamos la variable objetivo
# 1 = Psychic
# 0 = Fighting

df_filtrado['target'] = ((df_filtrado['Type 1'] == 'Psychic') | (df_filtrado['Type 2'] == 'Psychic')).astype(int)

print('\n' + '=' * 50)

print('\nDistribución de las clases:')

print(df_filtrado[['Type 1','Type 2']].stack().value_counts())
print('\n' + '=' * 50)
print('\n' + '=' * 50)
print("\nNúmero de Pokémon seleccionados:", len(df_filtrado))
print('\n' + '=' * 50)
display(df_filtrado)

***

En primer lugar, estudiaremos cómo se distribuyen las estadísticas dependiendo del tipo del pokémon para, de esta manera, ver si se aprecia alguna diferencia entre ellos.

***

In [ ]:
# Distribución de Pokémon por tipo
plt.figure(figsize=(7, 5))

sns.countplot(data=df_filtrado, x='target')

plt.title('Distribución de Pokémon por tipo')
plt.xticks([0, 1], ['Fighting', 'Psychic'])
plt.xlabel('Tipo')
plt.ylabel('Número de Pokémon')
plt.show()


# Comparación de las estadísticas según el tipo
for variable in estadísticas:

    plt.figure(figsize=(7, 5))

    sns.boxplot(data=df_filtrado, x='target', y=variable)
    plt.xticks([0, 1], ['Fighting', 'Psychic'])
    plt.xlabel('Tipo de Pokémon')
    plt.title(f'{variable} según el tipo de Pokémon')
    plt.show()

***
Empezamos a ver cosas interesantes. Atendiendo a las gráficas de ataque y ataque especial, vemos que sí hay una diferecia considerable.

A mayores podemos comprobar que tenemos bastantes más pokémon de tipo lucha que de tipo psíquico, además de que, como comentamos al principio, hay algunos que poseen ambos tipos a la vez.

***

***
Como hemos comentado anteriormente, para poder realizar el modelo de clasificación necesitamos dividir nuestros datos en varios conjuntos.

Primero, el conjunto de entrenamiento, que comprenderá los datos a partir de los cuales el modelo obtendrá las correlaciones que aplicará posteriormente. Este conjunto lo vamos a dividir en dos subconjuntos: uno para entrenar el algoritmo y otro para hacer una primera comprobación de su rendimiento.

Por último, vamos a escoger otro conjunto de prueba con el fin de evaluar cómo clasifica el modelo ante datos totalmente nuevos.

***

In [ ]:
from sklearn.preprocessing import StandardScaler

# Variables predictoras
X = df_filtrado[estadísticas].values

# Variable objetivo
y = df_filtrado['target'].values

# Primera división: entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# División del entrenamiento para obtener validación
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42, stratify=y_train)

# Escalado de las variables
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


print('\n' + '=' * 50)


print('\nDatos de entrenamiento:', X_train_scaled.shape)
print('Datos de validación:', X_val_scaled.shape)
print('Datos de prueba:', X_test_scaled.shape)

print('\n' + '=' * 50)

***

Hemos probado a variar los tamaños de los conjuntos de entrenamiento, validación y prueba para ver cómo esto afectaba al rendimiento del modelo. Intentamos escoger $\text{test\_size} = 0.42$, igual que en algunos programas de clase. Sin embargo, obtuvimos resultados peores debido a que perder elementos en el conjunto de entrenamiento suponía perder precisión.

***


****

### Construcción y entrenamiento de la red neuronal

Para construir nuestra red neuronal nos hemos inspirado en el modelo 'Detección de cáncer de mama' del aula virtual, debido a que nuestro tipo de clasificación es similar. De esta forma, creamos la siguiente arquitectura:

1. Una red de entrada de 20 neuronas tipo RELU
2. Capa de 20 neuronas tipo RELU
3. Capa de 20 neurones tipo RELU
4. Una última capa de una sola neurona tipo sigmoide

Escogeremos como función de pérdida '$\text{binary\_crossentropy}$' la cual es particularmente útil para clasificaciones en dos categorías.

****

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import SGD

from tensorflow.keras.callbacks import EarlyStopping

# Fijamos la semilla para reproducibilidad
np.random.seed(42) 
tf.random.set_seed(42)

# Creación de la red neuronal
model = keras.Sequential([layers.Dense(20, activation='relu', input_shape=(X_train_scaled.shape[1],)), layers.Dense(20, activation='relu'), layers.Dense(20, activation='relu'), layers.Dense(1, activation='sigmoid')])

# Compilación del modelo
sgd = SGD(learning_rate=0.01)

model.compile(optimizer=sgd, loss='binary_crossentropy', metrics=['accuracy'])

# Resumen de la arquitectura
model.summary()

In [ ]:
# Early Stopping para evitar sobreajuste
early_stop = EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True)

# Entrenamiento
print("Entrenando la red neuronal...")

history = model.fit(X_train_scaled, y_train, epochs=500, batch_size=10, validation_data=(X_val_scaled, y_val), callbacks=[early_stop], verbose=1)

print("\n¡Entrenamiento completado!")

print("Número de épocas ejecutadas:", len(history.history['loss']))

***

Una vez entrenado el modelo lo aplicaremos al conjunto de prueba. Para estudiar los resultados, mostraremos la precisión del modelo y la matriz de confusión con el objetivo de evaluar si hay un patrón.

***

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)

# Predicciones de la red neuronal
y_prob = model.predict(X_test_scaled)

# Convertimos probabilidades en clases: de esta forma las probabilidades mayores o iguales a 0.7 se consideran como clase 1 (Psíquico) y las menores como clase 0 (Lucha). Esto permite ajustar la sensibilidad del modelo según las necesidades del análisis.
y_hat_test = (y_prob >= 0.7).astype(int).ravel()

# Accuracy
print("Red Neuronal - Accuracy:", accuracy_score(y_test, y_hat_test))

# Informe de clasificación
print("\nInforme de clasificación:")

print(classification_report(y_test, y_hat_test, target_names=['Psychic', 'Fighting'], zero_division=0))

# Matriz de confusión
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_hat_test))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_hat_test)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Psychic', 'Fighting'])

disp.plot(cmap='Greens')

plt.title('Matriz de confusión de la red neuronal')
plt.show()

***

Hemos obtenido un valor de precisión considerablemente alto. Además, la matriz de confusión nos corrobora el correcto funcionamiento del modelo, el cual clasifica especialmente bien los tipo lucha.

Por lo tanto, efectivamente: ¡Hay patrones en algunos tipos de pokémon!

Cabe destacar el hecho de que es probable que se pueda obtener un mejor resultado si restringimos nuestra variable X a solo aquellas estadísticas que estén más diferenciadas entre si.

También podría ser interesante comprobar más estereotipos de cada tipo: si los voladores tienen mucha velocidad, si los tipo dragón tienen estadísiticas más altas que el resto, si los tipos roca y acero tienen más defensa que los demás... Hay muchos estereotipos que se tienen por ciertos a la hora de jugar que podrían ser estudiados.

***


***


Ahora, a modo de comprobación, representaremos las curvas de evolución de la precisión y la pérdida a lo largo de las épocas. Si no hemos sobreentrenado el modelo, la curva de pérdida de validación debería mantenerse decreciente.


***

In [ ]:
# Extraemos los datos del historial
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)


# Gráfica de Accuracy
plt.figure(figsize=(8, 5))

plt.plot(epochs, acc, label='Entrenamiento')

plt.plot(epochs, val_acc, label='Validación')

plt.title('Precisión de la red neuronal')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')

plt.legend()
plt.grid(True)
plt.show()


# Gráfica de Loss
plt.figure(figsize=(8, 5))

plt.plot(epochs, loss, label='Entrenamiento')

plt.plot(epochs, val_loss, label='Validación')

plt.title('Pérdida de la red neuronal')
plt.xlabel('Épocas')
plt.ylabel('Loss')

plt.legend()
plt.grid(True)
plt.show()

***

Vemos que la curva de pérdida efectivamente acaba en un mínimo: no hay sobreentrenamiento.

***

***

# Parte 3: Siempre buscamos los legendarios

Al jugar a la franquicia, lo habitual es prestar especial atención a los pokémon *legendarios*, ya que se presupone que poseen estadísticas muy superiores al resto y suelen protagonizar las carátulas de los videojuegos. Aprovechando que el conjunto de datos dispone de esta clasificación, analizaremos si los ejemplares con mejores estadísticas pertenecen exclusivamente a este grupo o si existen excepciones relevantes.

Estas excepciones pueden surgir porque una clasificación estrictamente binaria simplifica en exceso la realidad del juego, donde coexisten otras categorías con estadísticas muy altas, como los pokémon *singulares* o los *pseudolegendarios*.

Con todo, utilizaremos esta etiqueta para entrenar una nueva red neuronal con una arquitectura alternativa y evaluar su capacidad de clasificación.

***

In [ ]:
#Vamos a volver a cargar el dataset original, ya que lo habíamos filtrado.
df = pd.read_csv('trabajobio/Pokemon.csv')

print('\n' + '=' * 50)
display(df.head())
print('\n' + '=' * 50)

df['Legendary'].unique()


In [ ]:
#Vamos a poner los valores de la columna 'Legendary' como 0 y 1, en lugar de True y False, para poder trabajar con ellos en el modelo de red neuronal.

df['Legendary'] = df['Legendary'].astype(int) 

In [ ]:
#Vamos a escoger ahora como X ya no solo las estadísticas, sino también la estadística total, que vendría a ser la suma de todas ellas.

X = df[estadísticas + ['Total']].values
Y = df['Legendary'].values

# Primera división: entrenamiento y prueba
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42, stratify=Y)

# División del entrenamiento para obtener validación
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.20, random_state=42, stratify=Y_train)

# Escalado de las variables
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


print('\n' + '=' * 50)


print('\nDatos de entrenamiento:', X_train_scaled.shape)
print('Datos de validación:', X_val_scaled.shape)
print('Datos de prueba:', X_test_scaled.shape)

print('\n' + '=' * 50)

***

Creamos ahora el modelo que vamos a seguir. Esta vez vamos a hacer la siguiente red neuronal, probando a cambiar el tipo de activación:

1. Una neurona de input tipo RELU
2. Segunda cada de 64 neuronas tipo sigmoide
3. Una tercera etapa igual que la segunda
4. Una última de salida tipo RELU

***

In [ ]:
np.random.seed(42) 
tf.random.set_seed(42)

# Creación de la red neuronal
model = keras.Sequential([layers.Dense(1, activation='relu', input_shape=(X_train_scaled.shape[1],)), layers.Dense(64, activation='sigmoid'), layers.Dense(64, activation='sigmoid'), layers.Dense(1, activation='relu')])

# Compilación del modelo
sgd = SGD(learning_rate=0.01)

model.compile(optimizer=sgd, loss='binary_crossentropy', metrics=['accuracy'])


# Resumen de la arquitectura
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=40, restore_best_weights=True)

# Entrenamiento
print("Entrenando la red neuronal...")

history = model.fit(X_train_scaled, Y_train, epochs=500, batch_size=10, validation_data=(X_val_scaled, Y_val), callbacks=[early_stop], verbose=1)

print('\n¡Entrenamiento completado!')

print('\nNúmero de épocas ejecutadas:', len(history.history['loss']))

***

Vemos que se ejecutan pocas épocas por el hecho de que hay un porcentaje muy pequeño de legendarios, por lo que llegamos rápidamente al sobreentrenamiento.

***


In [ ]:
# Predicciones de la red neuronal
Y_prob = model.predict(X_test_scaled)

# Convertimos probabilidades en clases: de esta forma las probabilidades mayores o iguales a 0.7 se consideran como clase 1 (Psíquico) y las menores como clase 0 (Lucha). Esto permite ajustar la sensibilidad del modelo según las necesidades del análisis.
Y_hat_test = (Y_prob >= 0.7).astype(int).ravel()
# Accuracy
print("Red Neuronal - Accuracy:", accuracy_score(Y_test, Y_hat_test))

# Informe de clasificación
print("\nInforme de clasificación:")

print(classification_report(Y_test, Y_hat_test, target_names=['No legendario', 'Legendario'], zero_division=0))

# Matriz de confusión
print("\nMatriz de confusión:")
print(confusion_matrix(Y_test, Y_hat_test))

cm = confusion_matrix(Y_test, Y_hat_test)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No legendario', 'Legendario'])

disp.plot(cmap='Greens')

plt.title('Matriz de confusión de la red neuronal')
plt.show()


***

Aunque el modelo registra una precisión muy alta, su funcionamiento no es el adecuado. El motivo principal es la escasez de pokémon legendarios en el conjunto de datos. Al ser tan poco frecuentes, es probable que en las particiones de los datos apenas se incluyan ejemplos de esta clase, afectando negativamente al aprendizaje del modelo.

***

***
# Conclusión

Los resultados reflejan que *Game Freak* mantiene cierta coherencia en los parámetros de diseño de los tipos: los atributos de un pokémon parecen influir directamente en sus estadísticas base, un fenómeno especialmente visible en los tipos psíquico y lucha. Aunque a nivel general se asume que no existe una relación directa entre ambas variables, el análisis detallado demuestra que sí hay un impacto significativo.

Asimismo, la condición de "legendario" altera notablemente el potencial del pokémon. No obstante, el modelo podría presentar un sesgo en sus predicciones debido a la baja representatividad de pokémon legendarios en el dataset.

***

![Texto alternativo](https://media1.tenor.com/m/EjXQ9meeKxAAAAAd/sphealspin.gif)